# Vectors and rigid transforms

Points in these notebooks are row vectors with shape `(N, 3)`, and **Z is up**. A rotation `R` is active on column vectors, so `p' = R @ p + t`. For a whole cloud that is `(R @ points.T).T + t`, which is the same as `points @ R.T + t`.

State that convention before you write anything. Most bugs in this interview are a transpose, a sign, or a zero-length vector.

The solution cell replaces your function and runs the checks. Scroll past it until you have tried.


These cells are PyTorch. Geometry wants float64, so the setup cell calls `torch.set_default_dtype(torch.float64)`. The library default is float32, and a plane SVD or a Kabsch in float32 will miss a hand check that passes in float64.


In [ ]:
import torch

torch.set_default_dtype(torch.float64)
torch.set_printoptions(precision=4, sci_mode=False)


## 1. Angle and projection

Write `angles_and_projections`.

The angle between each row of `vectors` and `axis` is `arccos` of the cosine. Clamp the cosine to `[-1, 1]` before `arccos`, or floating-point error turns a parallel pair into NaN. A zero-length vector has an undefined angle; return 0 for that row.

The projection is the component of each vector along `axis`, as a vector. `axis` is not necessarily a unit vector.

Hand check: vectors `(1,0,0)`, `(0,1,0)`, `(1,1,0)` against axis `(2,0,0)` give angles `0`, `π/2`, `π/4` and projections `(1,0,0)`, `(0,0,0)`, `(1,0,0)`.

<details><summary>Hint</summary>

Normalize the axis once. Cosine is `(v · unit) / ||v||`. The projection of `v` onto `unit` is `(v · unit) * unit`.

</details>


In [ ]:
import torch

def angles_and_projections(vectors, axis):
    """
    Angles of vectors with axis, and their projections onto axis.

    vectors: (N, 3)
    axis: (3,)
    returns angles (N,) in radians, projections (N, 3)
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def angles_and_projections(vectors, axis):
    """Angles of vectors with axis, and their projections onto axis.

    vectors: (N, 3)
    axis: (3,)
    returns angles (N,) in radians, projections (N, 3)
    """
    vectors = _t(vectors)
    axis = _t(axis)
    axis_norm = torch.linalg.norm(axis)
    if axis_norm < 1e-12:
        raise ValueError("axis has zero length")
    unit = axis / axis_norm
    vec_norm = torch.linalg.norm(vectors, dim=1)
    safe = torch.clamp(vec_norm, min=1e-12)
    cos = torch.clamp((vectors @ unit) / safe, -1.0, 1.0)
    angles = torch.acos(cos)
    angles = torch.where(vec_norm < 1e-12, torch.zeros_like(angles), angles)
    projections = (vectors @ unit)[:, None] * unit
    return angles, projections

def check_angles_and_projections():
    vectors = torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [1.0, 1.0, 0.0], [0.0, 0.0, 0.0]])
    angles, proj = angles_and_projections(vectors, torch.tensor([2.0, 0.0, 0.0]))
    assert torch.allclose(angles[:3], torch.tensor([0.0, torch.pi / 2, torch.pi / 4]))
    assert float(angles[3]) == 0.0
    assert torch.allclose(proj, torch.tensor([[1, 0, 0], [0, 0, 0], [1, 0, 0], [0, 0, 0]], dtype=DTYPE))
    print("passed")

check_angles_and_projections()


## 2. Rotation about an arbitrary axis

Rotate an `(N, 3)` cloud about an axis through the origin, then add a translation.

Use Rodrigues' formula. Positive angles follow the right-hand rule: thumb along the axis, fingers curl toward the positive rotation. Return a proper rotation, so `R.T @ R = I` and `det(R) = +1`.

There are two equivalent formulas for a unit axis `k` and its skew matrix `K`:

- `R = I + sin(θ) K + (1 - cos(θ)) K²`
- `R = cos(θ) I + sin(θ) K + (1 - cos(θ)) (k kᵀ)`

`K²` and `k kᵀ` are different matrices. Mixing them is the usual bug.

Hand check: `(1,0,0)` rotated `+90°` about `+Z`, then translated by `(0,0,1)`, is `(0,1,1)`. A `+120°` rotation about `(1,1,1)` sends `(1,0,0)` to `(0,1,0)`.

<details><summary>Hint</summary>

`K = [[0, -kz, ky], [kz, 0, -kx], [-ky, kx, 0]]`. Normalize the axis before building `K`. Apply `R` to column vectors, then add `t`.

</details>


In [ ]:
import torch

def rotation_matrix(axis, theta):
    """
    Active rotation by `theta` radians about `axis` (Rodrigues).

    Positive theta follows the right-hand rule. The returned matrix acts on
    column vectors: p' = R @ p.
    """
    raise NotImplementedError

def transform_points(points, axis, theta, translation):
    """
    Rotate (N, 3) points about an axis through the origin, then translate.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def rotation_matrix(axis, theta):
    """Active rotation by `theta` radians about `axis` (Rodrigues).

    Positive theta follows the right-hand rule. p' = R @ p for a column vector.
    """
    axis = _t(axis)
    n = torch.linalg.norm(axis)
    if n < 1e-12:
        raise ValueError("axis has zero length")
    k = axis / n
    K = torch.tensor(
        [
            [0.0, -k[2], k[1]],
            [k[2], 0.0, -k[0]],
            [-k[1], k[0], 0.0],
        ],
        dtype=axis.dtype,
    )
    eye = torch.eye(3, dtype=axis.dtype)
    theta = torch.as_tensor(theta, dtype=axis.dtype)
    return eye + torch.sin(theta) * K + (1.0 - torch.cos(theta)) * (K @ K)

def transform_points(points, axis, theta, translation):
    """Rotate (N, 3) points about an axis through the origin, then translate."""
    points = _t(points)
    R = rotation_matrix(axis, theta)
    t = _t(translation)
    return (R @ points.T).T + t

def check_transform_points():
    out = transform_points([[1.0, 0.0, 0.0]], [0.0, 0.0, 1.0], torch.pi / 2, [0.0, 0.0, 1.0])
    assert torch.allclose(out, torch.tensor([[0.0, 1.0, 1.0]]), atol=1e-8)
    cycled = transform_points([[1.0, 0.0, 0.0]], [1.0, 1.0, 1.0], 2.0 * torch.pi / 3, [0.0, 0.0, 0.0])
    assert torch.allclose(cycled, torch.tensor([[0.0, 1.0, 0.0]]), atol=1e-8)
    back = transform_points([[1.0, 0.0, 0.0]], [0.0, 0.0, 1.0], -torch.pi / 2, [0.0, 0.0, 0.0])
    assert torch.allclose(back, torch.tensor([[0.0, -1.0, 0.0]]), atol=1e-8)
    images = transform_points(torch.eye(3), [0.2, -0.4, 0.5], 0.7, [0.0, 0.0, 0.0])
    R = images.T
    assert torch.allclose(R.T @ R, torch.eye(3, dtype=DTYPE), atol=1e-8)
    assert torch.allclose(torch.linalg.det(R), torch.tensor(1.0, dtype=DTYPE), atol=1e-8)
    print("passed")

check_transform_points()


## 3. Compose and invert rigid transforms

A rigid transform is `p' = R @ p + t`. Write three functions:

- `apply_rigid(points, R, t)` for an `(N, 3)` cloud
- `compose_rigid(R_first, t_first, R_second, t_second)` so the result applies `first`, then `second`
- `invert_rigid(R, t)` using `R.T`, not `torch.linalg.inv`

Hand check: start at `(1,0,0)`. Rotate `+90°` about Z and translate by `(1,0,0)`, then rotate `+90°` about Z again and translate by `(0,1,0)`. The point lands on `(-1, 2, 0)`. Applying the inverse returns the original point.

<details><summary>Hint</summary>

`R = R_second @ R_first` and `t = R_second @ t_first + t_second`. The inverse rotation is `R.T`, and the inverse translation is `-R.T @ t`.

</details>


In [ ]:
import torch

def apply_rigid(points, R, t):
    """
    p' = R @ p + t for row points of shape (N, 3).
    """
    raise NotImplementedError

def compose_rigid(R_first, t_first, R_second, t_second):
    """
    Rigid transform that applies `first`, then `second`.
    """
    raise NotImplementedError

def invert_rigid(R, t):
    """
    Inverse of p' = R @ p + t. Uses R.T, not a general inverse.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def compose_rigid(R_first, t_first, R_second, t_second):
    """Rigid transform that applies `first`, then `second`."""
    R = R_second @ R_first
    t = R_second @ _t(t_first) + _t(t_second)
    return R, t

def apply_rigid(points, R, t):
    """p' = R @ p + t for row points of shape (N, 3)."""
    points = _t(points)
    t = _t(t)
    return (R @ points.T).T + t

def invert_rigid(R, t):
    """Inverse of p' = R @ p + t. Uses R.T, not a general inverse."""
    t = _t(t)
    R_inv = R.T
    return R_inv, -R_inv @ t

def check_rigid():
    def rz(angle):
        c, s = torch.cos(torch.tensor(angle, dtype=DTYPE)), torch.sin(torch.tensor(angle, dtype=DTYPE))
        return torch.tensor([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]], dtype=DTYPE)

    R, t = compose_rigid(rz(torch.pi / 2), [1.0, 0.0, 0.0], rz(torch.pi / 2), [0.0, 1.0, 0.0])
    p = torch.tensor([[1.0, 0.0, 0.0]])
    assert torch.allclose(apply_rigid(p, R, t), torch.tensor([[-1.0, 2.0, 0.0]]), atol=1e-8)
    R_i, t_i = invert_rigid(R, t)
    assert torch.allclose(apply_rigid(apply_rigid(p, R, t), R_i, t_i), _t(p), atol=1e-8)
    c, s = torch.cos(torch.tensor(0.8)), torch.sin(torch.tensor(0.8))
    R = torch.tensor([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]]) @ torch.tensor(
        [[1.0, 0.0, 0.0], [0.0, c, -s], [0.0, s, c]]
    )
    t = torch.tensor([1.0, 2.0, -3.0])
    pts = torch.tensor([[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.2, -0.4, 0.6]])
    R_i, t_i = invert_rigid(R, t)
    assert torch.allclose(R_i @ R, torch.eye(3, dtype=DTYPE), atol=1e-8)
    assert torch.allclose(apply_rigid(apply_rigid(pts, R, t), R_i, t_i), _t(pts), atol=1e-8)
    print("passed")

check_rigid()


## 4. A frame for a cross-section

Survey geometry is Z-up. A road tangent lives in the XY plane. Build a right-handed frame whose **columns** are forward, left, and up.

- forward: the horizontal tangent, normalized, with z = 0
- up: `(0, 0, 1)`
- left: the horizontal direction such that `forward × left = up`

A zero-length tangent should raise `ValueError`. Do not assume the input is a unit vector.

Hand check: tangent `(3, 0)` returns `I`. Tangent `(0, 1)`, looking north, has left = west = `(-1, 0, 0)`.

<details><summary>Hint</summary>

If forward is `(tx, ty, 0)`, left is `(-ty, tx, 0)`. Check the frame by `R.T @ R ≈ I` and `det(R) ≈ +1`.

</details>


In [ ]:
import torch

def section_frame(tangent_xy):
    """
    Orthonormal frame whose columns are forward, left, up.

    tangent_xy: (2,) horizontal direction of travel, unnormalized.
    forward is that direction in the XY plane.
    up is +Z.
    left is horizontal so that forward x left = up (right-handed, Z-up).
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _cross(a, b):
    a, b = torch.broadcast_tensors(a, b)
    return torch.linalg.cross(a, b, dim=-1)

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def section_frame(tangent_xy):
    """Orthonormal frame whose columns are forward, left, up.

    tangent_xy: (2,) horizontal direction of travel.
    """
    tangent_xy = _t(tangent_xy).reshape(2)
    n = torch.linalg.norm(tangent_xy)
    if n < 1e-12:
        raise ValueError("tangent has zero horizontal length")
    forward = torch.stack((tangent_xy[0] / n, tangent_xy[1] / n, torch.zeros((), dtype=tangent_xy.dtype)))
    left = torch.stack((-forward[1], forward[0], torch.zeros((), dtype=tangent_xy.dtype)))
    up = torch.tensor([0.0, 0.0, 1.0], dtype=tangent_xy.dtype)
    return torch.column_stack((forward, left, up))

def check_section_frame():
    eye = section_frame(torch.tensor([3.0, 0.0]))
    assert torch.allclose(eye, torch.eye(3, dtype=DTYPE))
    north = section_frame(torch.tensor([0.0, 2.0]))
    assert torch.allclose(north[:, 0], torch.tensor([0.0, 1.0, 0.0]))
    assert torch.allclose(north[:, 1], torch.tensor([-1.0, 0.0, 0.0]))
    assert torch.allclose(torch.linalg.det(north), torch.tensor(1.0))
    diag = section_frame(torch.tensor([1.0, 1.0]))
    assert torch.allclose(diag.T @ diag, torch.eye(3, dtype=DTYPE), atol=1e-8)
    assert torch.allclose(_cross(diag[:, 0], diag[:, 1]), torch.tensor([0.0, 0.0, 1.0]), atol=1e-8)
    try:
        section_frame(torch.tensor([0.0, 0.0]))
        raise AssertionError("expected ValueError")
    except ValueError:
        pass
    print("passed")

check_section_frame()
